In [49]:
# ExtractCemeteries_JewishCounties.py
# 5/12/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(2, 4)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

# def extract_cemetery_data(first_elements):
#     global field_list_set,field_list
#     collection_df=pd.json_normalize(first_elements)
#     collection_columns_set=set(list(collection_df))
#     missing_columns=field_list_set-collection_columns_set
#     collection_df[list(missing_columns)]=np.nan
#     cemetery_info_df=collection_df[field_list]
#     return cemetery_info_df

def clean_dataframe(graves_df):
    global field_list_out
    for column in field_list_out:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list_out:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'ancestry-clientpath': 'findagrave-frontend',
  'apollographql-client-name': 'findagrave-cemetery-searchresults',
  'content-type': 'application/json',
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36'
}

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0


start_time=datetime.datetime.now()
event_time=start_time


process_name='Extract Cemeteries - Jewish Counties '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileIn="Jewish_Counties_Coordinates"
fileOutput="Cemeteries_Jewish_Counties"

if append==1:fileIn='Jewish_Counties_Coordinates_X'


fileCountiesIn=DirMain+"Data\\"+fileIn+".csv"
fileOut=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"OutputData\\Cemeteries\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_Interim.csv"

field_list=['cemeteryId','cemetery','city','county','state','countyId','stateId','lat', 'lon']
field_list_set=set(field_list)
field_list_out=['cemetery_name', 'city', 'county', 'state', 'cemetery_id', 'grave_count', 'cemetery_lat', 'cemetery_lon']

county_info_df=pd.read_csv(fileCountiesIn)

state_id_list=county_info_df["state_id"]
state_name_list=county_info_df["state_name"]
county_id_list=county_info_df["county_id"]
county_name_list=county_info_df["county_name"]
county_lat_list=county_info_df["county_lat"]
county_lon_list=county_info_df["county_lon"]

if append==1:print('**** Appending ****')    
if append!=1:cemetery_df=pd.DataFrame([])  

sleep_time0=5*60

url_prefix="https://www.findagrave.com/orc/graphql?operationName=getCemeteriesInBbox&variables=%7B%22search%22%3A%7B%22from%22%3A0%2C%22size%22%3A9999%2C%22boundingBox%22%3A%7B%22top_left%22%3A%7B%22lat%22%3A"
url_top_lat="%2C%22lon%22%3A"
url_top_lon="%7D%2C%22bottom_right%22%3A%7B%22lat%22%3A"
url_bottom_lat="%2C%22lon%22%3A"
url_bottom_lon="%7D%7D%7D%7D&extensions=%7B%22persistedQuery%22%3A%7B%22version%22%3A1%2C%22sha256Hash%22%3A%22edb98695eaea673b2c57597571c51d2477019b5bb547d350b50393d5b9ee6dd4%22%7D%7D"

expand_factor=.5
a=[1,-1,-1,1]
b=[1,-1,0,0]
c=[0,0,-1,1]
d=[1,0,0,1]
e=[0,-1,-1,0]
grid=[a,b,c,d,e]
coordinates_df = pd.DataFrame(grid, columns=['Top_Lat', 'Top_Lon', 'Bottom_Lat', 'Bottom_Lon'])

county_count=0
last_state='Alabama'
for county_id in county_id_list:
    cemetery_frame_df=pd.DataFrame([])
    memorial_count_df=pd.DataFrame([])
    cemetery_id_df=pd.DataFrame([])
    cemetery_info_df=pd.DataFrame([])
    cemetery_frame_pre_df=pd.DataFrame([])
    state_name=state_name_list.iloc[county_count]
    state_id=state_id_list.iloc[county_count]
    county_name=county_name_list.iloc[county_count]
    county_id=county_id_list.iloc[county_count]   
    county_lat=county_lat_list.iloc[county_count]
    county_lon=county_lon_list.iloc[county_count]
    county_count+=1
    # county_lat_top=county_lat+1.25
    # county_lon_top=county_lon-1.25
    # county_lat_bottom=county_lat-1.25
    # county_lon_bottom=county_lon+1.25

    cemetery_count=0
    for iTry in range(0,5):
        county_lat_top,county_lon_top,county_lat_bottom,county_lon_bottom=[county_lat,county_lon,county_lat,county_lon]+expand_factor*coordinates_df.iloc[iTry]    
        url=url_prefix+str(county_lat_top)+url_top_lat+str(county_lon_top)+url_top_lon+\
                str(county_lat_bottom)+url_bottom_lat+str(county_lon_bottom)+url_bottom_lon
    
        response = requests.request("GET", url, headers=my_header) 
        status_code=response.status_code
        url_source = response.text 
        
        if status_code!=200:print('\tstatus_code:',status_code);continue
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
        specific_tag_text = beautifulSoupText.get_text()
        capture_json=json.loads(specific_tag_text) 
  
        
        if len(pd.DataFrame(capture_json['data']['bulkCemeterySearch']))==0:print('county_id:',county_id,' Is Empty');continue
        collection2_df=pd.DataFrame(capture_json['data']['bulkCemeterySearch']['cemeteries'])
        first_elements = collection2_df['locations'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
        collection_df=pd.json_normalize(first_elements)
        collection_columns_set=set(list(collection_df))
        missing_columns=field_list_set-collection_columns_set
        collection_df[list(missing_columns)]=np.nan
        cemetery_info_df=collection_df
        # cemetery_info_df=pd.json_normalize(first_elements)[['cemetery','city','county','state']]
        cemetery_info_df[['cemetery_id','grave_count']]=collection2_df[['id','memorialCount']]
        cemetery_frame_df=cemetery_info_df.rename(columns={'cemetery': 'cemetery_name','stateId':'state_id','countyId':'county_id'}) 
        coordinate_vector=pd.json_normalize(collection2_df['coordinates'])
        cemetery_info_df[['cemetery_lat', 'cemetery_lon']]=coordinate_vector[['lat', 'lon']]        
        cemetery_frame_df=cemetery_info_df.rename(columns={'cemetery': 'cemetery_name','stateId':'state_id','countyId':'county_id'}) 
        cemetery_df = cemetery_df._append(cemetery_frame_df, ignore_index = True) 
        cemetery_unique_df = cemetery_df.drop_duplicates()
        cemetery_df=cemetery_unique_df   
        cemetery_count+=len(cemetery_frame_df)
        # if len(pd.DataFrame(capture_json['data']['bulkCemeterySearch']))!=9999:break

    end_time=datetime.datetime.now()
    run_processing_time=end_time-event_time
    event_time=end_time
    run_processing_time_seconds=int(run_processing_time.total_seconds())
    processing_time=end_time-start_time
    processing_time_minutes=int(processing_time.total_seconds())/60
    
    if last_state!=state_name:print('\n');last_state=state_name
    print(f"**** {process_name}  state: {state_name:<3} • county: {county_name:<20}• county count: {county_count}/{len(county_id_list)} •"+\
                  f"cemetery Count: {cemetery_count:<3} • Processing Time: {run_processing_time_seconds} Sec • Cum Processing Time {processing_time_minutes:.1f} min")
    cemetery_df=clean_dataframe(cemetery_df)
    cemetery_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    sleep_time=random_sleep(county_count+1)       
    time.sleep(sleep_time)


end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   

cemetery_df.to_csv(fileOut,mode="w", index=False,header=True)
cemetery_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Extract Cemeteries - Jewish Counties   *** , Time: 2025-05-12 11:46:32.899584  **

**** Extract Cemeteries - Jewish Counties   state: Alabama • county: Barbour County      • county count: 1/526 •cemetery Count: 1868 • Processing Time: 4 Sec • Cum Processing Time 0.1 min
**** Extract Cemeteries - Jewish Counties   state: Alabama • county: Colbert County      • county count: 2/526 •cemetery Count: 2996 • Processing Time: 7 Sec • Cum Processing Time 0.2 min
**** Extract Cemeteries - Jewish Counties   state: Alabama • county: Dallas County       • county count: 3/526 •cemetery Count: 2148 • Processing Time: 7 Sec • Cum Processing Time 0.3 min
**** Extract Cemeteries - Jewish Counties   state: Alabama • county: Etowah County       • county count: 4/526 •cemetery Count: 2551 • Processing Time: 8 Sec • Cum Processing Time 0.4 min
**** Extract Cemeteries - Jewish Counties   state: Alabama • county: Jefferson County    • county count: 5/526 •cemetery Count: 2284 • Processing Tim

In [ ]:
def extract_cemetery_data(first_elements):
    global field_list_set,field_list
    collection_df=pd.json_normalize(first_elements)
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    first_elements=collection_df[field_list]
    return first_elements



field_list=['cemeteryId','cemetery','city','county','state','countyId','stateId','lat', 'lon']

field_list_set=set(field_list)


collection2_df=pd.DataFrame(capture_json['data']['bulkCemeterySearch']['cemeteries'])
first_elements = collection2_df['locations'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
first_elements=extract_cemetery_data(first_elements)
cemetery_info_df=pd.json_normalize(first_elements)[['cemetery','city','county','state']]
cemetery_info_df=collection2_df[['id','memorialCount']]
coordinate_vector=pd.json_normalize(collection2_df['coordinates'])

cemetery_info_df[['cemetery_lat', 'cemetery_lon']]=coordinate_vector[['lat', 'lon']]
# cemetery_frame_df=cemetery_info_df.rename(columns={'cemetery': 'cemetery_name','stateId':'state_id','countyId':'county_id'}) 
# cemetery_df = cemetery_df._append(cemetery_frame_df, ignore_index = True) 
# cemetery_unique_df = cemetery_df.drop_duplicates()
# cemetery_df=cemetery_unique_df   
# cemetery_count+=len(cemetery_frame_df)
      
cemetery_info_df
list(coordinate_vector)

# collection_df=pd.json_normalize(first_elements)
# collection_columns_set=set(list(collection_df))
# missing_columns=field_list_set-collection_columns_set
# collection_df[list(missing_columns)]=np.nan
# first_elements=collection_df[field_list]
# first_elements

In [48]:
collection_df=pd.json_normalize(first_elements)
collection_columns_set=set(list(collection_df))
missing_columns=field_list_set-collection_columns_set
collection_df[list(missing_columns)]=np.nan
collection_df

,language,cemeteryId,cemetery,cemeteryAbbreviation,countyId,county,countyAbbreviation,stateId,state,stateAbbreviation,...,country,countryAbbreviation,continentId,continent,continentAbbreviation,cityId,city,cityAbbreviation,lon,lat
0,en,cemetery_2785394,Warrior Cemetery,None,county_88,Tuscaloosa County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,NaN,NaN,NaN,NaN,NaN
1,en,cemetery_2791186,Davis,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_7019,McCalla,NaN,NaN,NaN
2,en,cemetery_2796021,Dynasty,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_5116,Dolomite,NaN,NaN,NaN
3,en,cemetery_2819355,Badger fementary,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_408265,Adger,NaN,NaN,NaN
4,en,cemetery_2473104,Smithville Baptist Church Cemetery,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_3998,Bessemer,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,en,cemetery_25077,Oakland Cemetery,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_4055,Birmingham,NaN,NaN,NaN
202,en,cemetery_25688,Prudes Creek Baptist Church Cemetery,None,county_88,Tuscaloosa County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,NaN,NaN,NaN,NaN,NaN
203,en,cemetery_22987,George Washington Carver Memorial Gardens,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_7175,Minor,NaN,NaN,NaN
204,en,cemetery_25436,Pine Hill Cemetery,None,county_62,Jefferson County,None,state_3,Alabama,AL,...,United States of America,USA,continent_68,North America,None,city_3998,Bessemer,NaN,NaN,NaN


In [ ]:
list(collection2_df)

In [ ]:
list(first_elements)
# ,'cemetery_lat','cemetery_lon'

In [ ]:
list(collection2_df)#[['id','coordinates.lat','coordinates.lon']]
# collection2_df['coordinates.lat'].lat

collection2_df=pd.DataFrame(capture_json['data']['bulkCemeterySearch']['cemeteries'])
coordinate_vector=pd.json_normalize(collection2_df['coordinates'])
cemetery_info_df[['cemetery_lat', 'cemetery_lon']]=coordinate_vector[['lat', 'lon']]
cemetery_info_df